In [4]:
from pathlib import Path
import shutil

PROJECT = Path("/mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey")

# 기존 1배율 산출물
OLD_DATA = Path("/mnt/d/LJH/data/final_data_100k_64.parquet")
OLD_JSON = PROJECT / "predict" / "result_xgb.json"

# sweep용 경로
SWEEP_DATA = PROJECT / "predict" / "tau_sweep_data"
SWEEP_RESULTS = PROJECT / "predict" / "tau_sweep_results"

assert OLD_DATA.exists(), OLD_DATA
assert OLD_JSON.exists(), OLD_JSON

SWEEP_DATA.mkdir(parents=True, exist_ok=True)
SWEEP_RESULTS.mkdir(parents=True, exist_ok=True)

shutil.copy2(OLD_DATA, SWEEP_DATA / "final_data_1_00x.parquet")
shutil.copy2(OLD_JSON, SWEEP_RESULTS / "result_xgb_1_00x.json")

print("1.00x 기존 데이터·결과를 sweep 경로에 복사 완료")

1.00x 기존 데이터·결과를 sweep 경로에 복사 완료


In [7]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT = Path("/mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey")

SCRIPT = PROJECT / "run_xgb_tau_sweep.py"
PREPROCESSOR = PROJECT / "preprocessing_fast.py"
WORKER = PROJECT / "tau_row_group_worker.py"

assert SCRIPT.exists(), SCRIPT
assert PREPROCESSOR.exists(), PREPROCESSOR
assert WORKER.exists(), WORKER

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TAU_PREPROCESS_PATH"] = str(PREPROCESSOR)

# 모든 새 배율은 기존 1배율 행을 기준으로 생성
env["TAU_REFERENCE_1X_PARQUET"] = str(OLD_DATA)

process = subprocess.Popen(
    [sys.executable, "-u", str(SCRIPT)],
    cwd=str(PROJECT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

exit_code = process.wait()
print(f"\n[FINISHED] exit code = {exit_code}")

if exit_code != 0:
    raise RuntimeError(f"Tau sweep failed: exit code {exit_code}")

[TAU PREPROCESSOR] Using: /mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey/preprocessing_fast.py

===== 0.50x: SMS=36.0h / Email=24.0h / Push=12.0h =====
[SKIP ALL] 전처리 데이터와 XGBoost 결과가 모두 존재: 0.50x
  [DATA]   /mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey/predict/tau_sweep_data/final_data_0_50x.parquet
  [RESULT] /mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey/predict/tau_sweep_results/result_xgb_0_50x.json

===== 0.75x: SMS=54.0h / Email=36.0h / Push=18.0h =====
[SKIP ALL] 전처리 데이터와 XGBoost 결과가 모두 존재: 0.75x
  [DATA]   /mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey/predict/tau_sweep_data/final_data_0_75x.parquet
  [RESULT] /mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey/predict/tau_sweep_results/result_xgb_0_75x.json

===== 1.00x: SMS=72.0h / Email=48.0h / Push=24.0h =====
[SKIP] 이미 완료된 전처리 사용: /mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey/predict/tau_sweep_data/final_data_1_00x.parquet
[DEVICE] Detected device for XGBoost: cuda
[LOAD] Loading dataset from: /mnt/c/Users/DIVE_GUEST/LJH/ecommerce

In [10]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT = Path("/mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey")
SCRIPT = PROJECT / "predict" / "compute_tau_targeting_metrics.py"

assert SCRIPT.exists(), SCRIPT

process = subprocess.Popen(
    [sys.executable, "-u", str(SCRIPT)],
    cwd=str(SCRIPT.parent),
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

exit_code = process.wait()
print(f"\n[FINISHED] exit code = {exit_code}")

if exit_code != 0:
    raise RuntimeError(f"Targeting-metric calculation failed: {exit_code}")


===== 0.50x: targeting metrics =====
  [LOAD] fold 0: xgb_all_fold0.pkl
/home/dive_guest/.conda/envs/ljh_312/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [12:00:26] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
  [LOAD] fold 1: xgb_all_fold1.pkl
  [LOAD] fold 2: xgb_all_fold2.pkl
  Top 1,000: TP=494 | P@K=49.4000% | Recall=26.6883% | Lift=400.30x
  Top 2,000: TP=724 | P@K=36.2000% | Recall=39.1140% | Lift=293.34x
  Top 3,000: TP=940 | P@K=31.3333% | Recall=50.7834% | Lift=253.90x
  Top 5,000: TP=1,270 | P@K=25.4000% | Recall=68.6116% | Lift=205.82x
  

In [6]:
from pathlib import Path
from datetime import datetime

PROJECT = Path("/mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey")
bad_json = (
    PROJECT / "predict" / "tau_sweep_results" / "result_xgb_1_00x.json"
)

assert bad_json.exists(), bad_json

backup = bad_json.with_name(
    f"{bad_json.stem}.invalid_{datetime.now():%Y%m%d_%H%M%S}.json"
)
bad_json.rename(backup)

print("깨진 기존 JSON 백업:", backup)
print("1.00x 데이터는 유지됨. 다음 실행에서 XGBoost만 다시 평가됩니다.")

깨진 기존 JSON 백업: /mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey/predict/tau_sweep_results/result_xgb_1_00x.invalid_20260813_230614.json
1.00x 데이터는 유지됨. 다음 실행에서 XGBoost만 다시 평가됩니다.


In [4]:
import os
import sys
import json
import subprocess
import pandas as pd
import numpy as np

# 1. openpyxl 라이브러리 자동 임포트/설치
try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
except ImportError:
    print("[*] openpyxl 라이브러리 설치 중...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

# 2. 다중 JSON 디코딩 (Extra data 에러 방지)
def load_json_file(filepath):
    if not os.path.exists(filepath):
        return {}
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read().strip()
    if not content:
        return {}
    try:
        return json.loads(content)
    except Exception:
        pass
        
    decoder = json.JSONDecoder()
    pos = 0
    merged_data = {}
    while pos < len(content):
        while pos < len(content) and content[pos] in " \t\r\n":
            pos += 1
        if pos >= len(content):
            break
        try:
            obj, end_pos = decoder.raw_decode(content, pos)
            if isinstance(obj, dict):
                merged_data.update(obj)
            pos = end_pos
        except Exception:
            next_brace = content.find('{', pos + 1)
            if next_brace == -1:
                break
            pos = next_brace
    return merged_data

# 3. JSON 파일 경로 자동 탐색
def find_result_files():
    search_dirs = [
        r"D:\LJH\predict",
        r"/mnt/d/LJH/predict",
        r"C:\Users\DIVE_GUEST\LJH\predict",
        r"c:\Users\user\Desktop\마케팅도메인지식기반 구매예측_논문\장바구니 이탈 연구 선행 논문\새 폴더\predict",
        r"/mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey/results",
        os.getcwd(),
        os.path.join(os.getcwd(), "predict"),
        os.path.join(os.getcwd(), "results")
    ]
    file_map = {}
    model_keys = {
        "XGB": ["result_xgb.json", "result_XGB.json"],
        "RF": ["result_RF.json", "result_rf.json"],
        "MLP": ["result_MLP.json", "result_mlp.json"],
        "CNN": ["result_CNN.json", "result_cnn.json"],
        "RNN": ["result_RNN.json", "result_rnn.json"],
        "LSTM": ["result_LSTM.json", "result_lstm.json"],
        "CNN-LSTM": ["result_CNNLSTM.json", "result_cnnlstm.json"],
        "RNN-LSTM": ["result_RNNLSTM.json", "result_rnnlstm.json"],
        "TabNet": ["result_tabnet.json", "result_TabNet.json"]
    }
    for algo, filenames in model_keys.items():
        found = None
        for d in search_dirs:
            if not os.path.exists(d):
                continue
            for fname in filenames:
                full_path = os.path.join(d, fname)
                if os.path.exists(full_path):
                    found = full_path
                    break
            if found:
                break
        if found:
            file_map[algo] = found
    return file_map

# 4. 상대적 증감률 계산
def calc_pct_change(base_val, total_val):
    if base_val is None or total_val is None or base_val == 0:
        return ""
    pct = ((total_val - base_val) / base_val) * 100.0
    sign = "+" if pct >= 0 else ""
    return f" ({sign}{pct:.1f}%)"

# 5. 서식 적용된 엑셀 파일 생성
def create_styled_excel():
    file_map = find_result_files()
    print(f"[*] 결과 파일 로드 완료: {list(file_map.keys())}")
    
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "표3_성능비교"
    ws.views.sheetView[0].showGridLines = True
    
    # 1) 제목 행 (A1:I1 병합)
    title_text = "<표 3> 원분포 홀드아웃 테스트 세트(0.1234%)에서의 알고리즘별 성능 비교"
    ws.merge_cells("A1:I1")
    cell_a1 = ws["A1"]
    cell_a1.value = title_text
    cell_a1.font = Font(name="맑은 고딕", size=14, bold=True)
    cell_a1.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 32
    
    # 2) 헤더 행 (진한 회색 배경)
    headers = ["Algorithm", "Feature Set", "Precision", "Recall", "F1-score", "Accuracy", "ROC-AUC", "F2-score", "PR-AUC"]
    header_row_idx = 2
    ws.row_dimensions[header_row_idx].height = 28
    
    header_fill = PatternFill(start_color="A6A6A6", end_color="A6A6A6", fill_type="solid")
    header_font = Font(name="맑은 고딕", size=11, bold=True, color="000000")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    
    thin_side = Side(border_style="thin", color="000000")
    medium_side = Side(border_style="medium", color="000000")
    
    for col_idx, h_text in enumerate(headers, 1):
        cell = ws.cell(row=header_row_idx, column=col_idx, value=h_text)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = header_align
        cell.border = Border(top=medium_side, bottom=medium_side, left=thin_side, right=thin_side)
        
    # 3) 데이터 행 및 모델별 2행 수직 셀 병합 서식
    algo_order = ["XGB", "RF", "MLP", "CNN", "RNN", "LSTM", "CNN-LSTM", "RNN-LSTM", "TabNet"]
    metrics_keys = [
        ("Precision", "precision"),
        ("Recall", "recall"),
        ("F1-score", "f1_score"),
        ("Accuracy", "accuracy"),
        ("ROC-AUC", "roc_auc"),
        ("F2-score", "f2_score"),
        ("PR-AUC", "pr_auc")
    ]
    
    algo_fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
    algo_font = Font(name="맑은 고딕", size=11, bold=True)
    data_font = Font(name="맑은 고딕", size=10)
    
    current_row = 3
    
    for algo_idx, algo in enumerate(algo_order):
        filepath = file_map.get(algo)
        data = load_json_file(filepath) if filepath else {}
        base_metrics = data.get("base", {})
        total_metrics = data.get("all", {})
        
        row_base_idx = current_row
        row_total_idx = current_row + 1
        
        ws.row_dimensions[row_base_idx].height = 22
        ws.row_dimensions[row_total_idx].height = 24
        
        # Algorithm 2줄 셀 병합 및 연한 회색 배경
        ws.merge_cells(start_row=row_base_idx, start_column=1, end_row=row_total_idx, end_column=1)
        algo_cell = ws.cell(row=row_base_idx, column=1, value=algo)
        algo_cell.font = algo_font
        algo_cell.fill = algo_fill
        algo_cell.alignment = Alignment(horizontal="center", vertical="center")
        ws.cell(row=row_total_idx, column=1).fill = algo_fill
        
        # Feature Set (Baseline / Total)
        ws.cell(row=row_base_idx, column=2, value="Baseline").alignment = Alignment(horizontal="center", vertical="center")
        ws.cell(row=row_total_idx, column=2, value="Total").alignment = Alignment(horizontal="center", vertical="center")
        ws.cell(row=row_base_idx, column=2).font = data_font
        ws.cell(row=row_total_idx, column=2).font = data_font
        
        # 수치 입력 및 증감률 서식
        for col_idx, (col_name, key_name) in enumerate(metrics_keys, 3):
            b_val = base_metrics.get(key_name, None)
            t_val = total_metrics.get(key_name, None)
            
            b_str = f"{b_val:.4f}" if b_val is not None else "-"
            if t_val is not None:
                pct_str = calc_pct_change(b_val, t_val)
                t_str = f"{t_val:.4f}{pct_str}"
            else:
                t_str = "-"
                
            c_base = ws.cell(row=row_base_idx, column=col_idx, value=b_str)
            c_tot  = ws.cell(row=row_total_idx, column=col_idx, value=t_str)
            
            c_base.font = data_font
            c_tot.font  = data_font
            c_base.alignment = Alignment(horizontal="center", vertical="center")
            c_tot.alignment  = Alignment(horizontal="center", vertical="center")
            
        # 테두리 설정
        is_last_algo = (algo_idx == len(algo_order) - 1)
        bottom_style = medium_side if is_last_algo else thin_side
        
        for r_idx in [row_base_idx, row_total_idx]:
            for c_idx in range(1, 10):
                cell = ws.cell(row=r_idx, column=c_idx)
                b_bottom = thin_side if r_idx == row_base_idx else bottom_style
                cell.border = Border(top=thin_side, bottom=b_bottom, left=thin_side, right=thin_side)
                
        current_row += 2
        
    # 열 너비 자동 설정
    ws.column_dimensions['A'].width = 16
    ws.column_dimensions['B'].width = 14
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 22
    ws.column_dimensions['E'].width = 22
    ws.column_dimensions['F'].width = 20
    ws.column_dimensions['G'].width = 20
    ws.column_dimensions['H'].width = 20
    ws.column_dimensions['I'].width = 22
    
    out_dir = os.path.dirname(list(file_map.values())[0]) if file_map else os.getcwd()
    output_excel_path = os.path.join(out_dir, "표3_원분포_홀드아웃_테스트세트_성능비교.xlsx")
    wb.save(output_excel_path)
    print(f"\n[★ 성공] 이미지 논문 양식과 동일한 엑셀 파일 생성 완료:")
    print(f" -> {output_excel_path}")
    return output_excel_path

# 즉시 실행
create_styled_excel()


[*] 결과 파일 로드 완료: ['XGB', 'RF', 'MLP', 'CNN', 'RNN', 'LSTM', 'CNN-LSTM', 'RNN-LSTM', 'TabNet']

[★ 성공] 이미지 논문 양식과 동일한 엑셀 파일 생성 완료:
 -> /mnt/d/LJH/predict/표3_원분포_홀드아웃_테스트세트_성능비교.xlsx


'/mnt/d/LJH/predict/표3_원분포_홀드아웃_테스트세트_성능비교.xlsx'

In [11]:
import json
from pathlib import Path

from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


SOURCE = Path(r"/mnt/d/LJH/predict/result_xgb.json")
OUTPUT = Path("/mnt/d/LJH/predict/도메인_지식축별_실험결과표.xlsx")

LABELS = {
    "base": "Baseline",
    "v0_v1": "Baseline\n+ ① 구매/재구매 주기",
    "v0_v2": "Baseline\n+ ② 시계열적 반응",
    "v0_v3": "Baseline\n+ ③ 마케팅 피로도",
    "v0_v4": "Baseline\n+ ④ 마케팅 채널 효과성",
}
ORDER = list(LABELS)

thin = Side(style="thin", color="000000")
medium = Side(style="medium", color="000000")
thick = Side(style="thick", color="000000")


def title(ws, text, end_col):
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=end_col)
    c = ws.cell(1, 1, text)
    c.font = Font(name="맑은 고딕", size=18)
    c.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 36


def frame_table(ws, end_col, end_row, gray_from=3):
    for col in range(1, end_col + 1):
        c = ws.cell(3, col)
        c.fill = PatternFill("solid", fgColor="D9D9D9" if col >= gray_from else "F2F2F2")
        c.font = Font(name="Times New Roman" if col > 1 else "맑은 고딕", size=12, bold=True)
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border = Border(top=thick, bottom=thick, left=thin if col > 1 else None,
                          right=thin if col < end_col else None)
    ws.row_dimensions[3].height = 64
    for row in range(4, end_row + 1):
        for col in range(1, end_col + 1):
            c = ws.cell(row, col)
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            c.font = Font(name="맑은 고딕", size=11, bold=(col == 1))
            c.border = Border(top=thin if row > 4 else None,
                              bottom=thick if row == end_row else None,
                              left=thin if col > 1 else None,
                              right=thin if col < end_col else None)
        ws.row_dimensions[row].height = 54 if row > 4 else 28
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = "B4"
    ws.page_setup.orientation = "landscape"
    ws.page_setup.fitToWidth = 1
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.print_area = f"A1:{get_column_letter(end_col)}{end_row}"


def make_table4(wb, data):
    ws = wb.active
    ws.title = "표 4_증분 기여도"
    title(ws, "<표 4> 최적 알고리즘(XGBoost) 기반 도메인 지식 축별 증분 기여도", 9)
    headers = ["Experiment", "Feature\nCount", "Δ\nPrecision", "Δ\nRecall", "Δ\nF1-score",
               "Δ\nAccuracy", "Δ\nROC-AUC", "Δ\nF2-score", "Δ\nPR-AUC"]
    for col, h in enumerate(headers, 1):
        ws.cell(3, col, h)
    metric_keys = ["precision", "recall", "f1_score", "accuracy", "roc_auc", "f2_score", "pr_auc"]
    base = data["base"]
    for row, key in enumerate(ORDER, 4):
        d = data[key]
        ws.cell(row, 1, LABELS[key])
        ws.cell(row, 2, d["num_features"])
        for col, metric in enumerate(metric_keys, 3):
            # Baseline은 원본값, 추가 행은 "원본값 (Baseline 대비 상대 증감률%)"
            if key == "base":
                value = f"{base[metric]:.3f}"
            else:
                raw = d[metric]
                change = (raw - base[metric]) / base[metric] * 100
                value = f"{raw:.3f} ({change:+.3f}%)"
            ws.cell(row, col, value)
    ws.column_dimensions["A"].width = 30
    ws.column_dimensions["B"].width = 13
    for col in range(3, 10):
        ws.column_dimensions[get_column_letter(col)].width = 15
    frame_table(ws, 9, 8)
    ws["A10"] = "주: 추가 행은 각 지표의 원본값과 Baseline 대비 상대 증감률(%)을 함께 표시함."
    ws["A10"].font = Font(name="맑은 고딕", size=9, italic=True)
    ws.merge_cells("A10:I10")


def make_table5(wb, data):
    ws = wb.create_sheet("표 5_Top-K 및 오차")
    title(ws, "<표 5> 도메인 지식 축별 Top-K 타겟팅 적중 수 및 상세 오차 행렬", 8)
    headers = ["Experiment", "Top 1k\nHits", "Top 2k\nHits", "Top 3k\nHits",
               "진짜 구매\n(TP)", "거짓 정보\n(FP)", "미포착\n(FN)", "진짜 비구매\n(TN)"]
    for col, h in enumerate(headers, 1):
        ws.cell(3, col, h)
    for row, key in enumerate(ORDER, 4):
        d = data[key]
        cm = d["confusion_matrix"]
        vals = [LABELS[key], d["top1000_hits"], d["top2000_hits"], d["top3000_hits"],
                cm["tp"], cm["fp"], cm["fn"], cm["tn"]]
        for col, value in enumerate(vals, 1):
            ws.cell(row, col, value)
            if col > 1:
                ws.cell(row, col).number_format = "#,##0"
    ws.column_dimensions["A"].width = 30
    for col in range(2, 9):
        ws.column_dimensions[get_column_letter(col)].width = 16
    frame_table(ws, 8, 8, gray_from=9)


def make_raw(wb, data):
    ws = wb.create_sheet("원본 데이터")
    metrics = ["num_features", "optimal_threshold", "accuracy", "precision", "recall", "f1_score",
               "f2_score", "roc_auc", "pr_auc", "top1000_hits", "top2000_hits", "top3000_hits",
               "tp", "fp", "fn", "tn"]
    ws.append(["experiment"] + metrics)
    for key, d in data.items():
        cm = d["confusion_matrix"]
        values = [d.get(m, cm.get(m)) for m in metrics]
        ws.append([key] + values)
    for c in ws[1]:
        c.font = Font(bold=True)
        c.fill = PatternFill("solid", fgColor="D9EAF7")
        c.alignment = Alignment(horizontal="center")
    ws.freeze_panes = "B2"
    ws.auto_filter.ref = ws.dimensions
    for col in range(1, len(metrics) + 2):
        ws.column_dimensions[get_column_letter(col)].width = 18


def main():
    data = json.loads(SOURCE.read_text(encoding="utf-8-sig"))
    wb = Workbook()
    make_table4(wb, data)
    make_table5(wb, data)
    make_raw(wb, data)
    wb.save(OUTPUT)
    print(OUTPUT)


if __name__ == "__main__":
    main()



/mnt/d/LJH/predict/도메인_지식축별_실험결과표.xlsx


In [2]:
import os, sys, random, json, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve
)

import xgboost as xgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

# =====================================================
# 1. CONFIG & SEED
# =====================================================
SEED = 1
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

BASE_DIR = r"C:\Users\DIVE_GUEST\LJH\ecommerce_journey"
if not os.path.exists(BASE_DIR) and os.path.exists(r"/mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey"):
    BASE_DIR = r"/mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey"
if not os.path.exists(BASE_DIR):
    BASE_DIR = r"c:\Users\user\Desktop\마케팅도메인지식기반 구매예측_논문\장바구니 이탈 연구 선행 논문"

NEW_FOLDER = os.path.join(BASE_DIR, "새 폴더")

DATA_DIR = r"/mnt/d/LJH/data"
if not os.path.exists(DATA_DIR) and os.path.exists(r"D:\LJH\data"):
    DATA_DIR = r"D:\LJH\data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = BASE_DIR

PATH_PARQUET = os.path.join(DATA_DIR, "final_data_100k_64.parquet")
if not os.path.exists(PATH_PARQUET):
    PATH_PARQUET = os.path.join(BASE_DIR, "final_data_100k_64.parquet")



N_TRIALS = 15
TEST_FRAC = 0.15

try:
    _dummy = xgb.XGBClassifier(device="cuda", tree_method="hist", n_estimators=1)
    _dummy.fit(np.array([[0.0]], dtype=np.float32), np.array([0]))
    DEVICE = "cuda"
except Exception:
    DEVICE = "cpu"

print(f"[DEVICE] Detected device for XGBoost: {DEVICE}")

# =======================================================
# 2. FEATURE GROUPS (v0 ~ v4) FOR ABLATION STUDY
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

cols_v1 = ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']
cols_v2 = ['cal_is_weekend', 'cal_week_of_month', 'feat_dow_shift', 'feat_eoq_bump', 'feat_hour_shift', 'feat_payday_bump']
cols_v3 = ['ctx_tc_open_rate_30d', 'feat_fatigue', 'feat_last_any_hours', 'feat_last_email_hours', 'feat_last_mobile_push_hours', 'u_cadence_std_30d', 'u_click_rate_30d', 'u_open_cnt_30d', 'u_open_rate_30d']
cols_v4 = ['feat_like_last_success', 'feat_path_align', 'feat_topic_novelty', 'topic_N7', 'topic_t_since_hours']



# =====================================================
# 3. LOAD DATASET
# =====================================================
if not os.path.exists(PATH_PARQUET):
    print(f"[ERROR] Parquet file does not exist at {PATH_PARQUET}!")
    sys.exit(1)

print(f"[LOAD] Loading dataset from: {PATH_PARQUET}")
df = pd.read_parquet(PATH_PARQUET)
TARGET = "is_purchased"

# Memory Downcasting (Reduces RAM from 12.5GB to 3.2GB!)
print("[MEMORY OPTIMIZATION] Downcasting feature dtypes to float32/int16...")
for c in df.columns:
    if "float" in str(df[c].dtype):
        df[c] = df[c].astype(np.float32)
    elif "int" in str(df[c].dtype) and c != TARGET:
        df[c] = df[c].astype(np.int16)
gc.collect()

print(f"[DATASET CHECK] Total Rows: {len(df):,d} | Positives: {(df[TARGET]==1).sum():,d} | Target Imbalance Ratio: {df[TARGET].mean()*100:.6f}%")

def get_existing_cols(df, col_list):
    return [c for c in col_list if c in df.columns]

feat_cols_base  = get_existing_cols(df, cols_v0)
feat_cols_v0_v1 = get_existing_cols(df, cols_v0 + cols_v1)
feat_cols_v0_v2 = get_existing_cols(df, cols_v0 + cols_v2)
feat_cols_v0_v3 = get_existing_cols(df, cols_v0 + cols_v3)
feat_cols_v0_v4 = get_existing_cols(df, cols_v0 + cols_v4)
feat_cols_all   = get_existing_cols(df, cols_v0 + cols_v1 + cols_v2 + cols_v3 + cols_v4)

# =====================================================
# 4. HELPER FUNCTIONS
# =====================================================
def find_best_f2_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    if len(thresholds) == 0:
        return 0.5, 0.0
    f2_scores = (5 * precisions * recalls) / np.maximum(4 * precisions + recalls, 1e-10)
    best_idx = np.argmax(f2_scores)
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f2 = f2_scores[best_idx]
    return float(best_thr), float(best_f2)

def find_best_f1_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    if len(thresholds) == 0:
        return 0.5, 0.0
    f1_scores = 2 * (precisions * recalls) / np.maximum(precisions + recalls, 1e-10)
    best_idx = np.argmax(f1_scores)
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f1 = f1_scores[best_idx]
    return float(best_thr), float(best_f1)


# =====================================================
# 5. INDIVIDUAL OPTUNA & ABLATION EVALUATION
# =====================================================
ablation_results = {}

experiments = [
    ("base",  feat_cols_base),
    ("v0_v1", feat_cols_v0_v1),
    ("v0_v2", feat_cols_v0_v2),
    ("v0_v3", feat_cols_v0_v3),
    ("v0_v4", feat_cols_v0_v4),
    ("all",   feat_cols_all),
]

all_indices = np.arange(len(df))
y_all = df[TARGET].values.astype(np.int32)

idx_tr_full, idx_te = train_test_split(all_indices, test_size=TEST_FRAC, stratify=y_all, random_state=SEED)
idx_tr, idx_va = train_test_split(idx_tr_full, test_size=0.17647, stratify=y_all[idx_tr_full], random_state=SEED)

raw_ratio = (y_all[idx_tr] == 0).sum() / float(max(1, (y_all[idx_tr] == 1).sum()))
print(f"[COST-SENSITIVE] Full Train Set Negative/Positive Ratio = {raw_ratio:.2f}")

def get_memory_safe_subsample(X_in, Y_in, max_negs=250_000, seed=SEED):
    pos_idx = np.where(Y_in == 1)[0]
    neg_idx = np.where(Y_in == 0)[0]
    if len(neg_idx) <= max_negs:
        return X_in, Y_in
    rng_sub = np.random.RandomState(seed)
    sub_neg_idx = rng_sub.choice(neg_idx, size=max_negs, replace=False)
    comb_idx = np.concatenate([pos_idx, sub_neg_idx])
    rng_sub.shuffle(comb_idx)
    return X_in[comb_idx], Y_in[comb_idx]

for exp_idx, (exp_name, feat_cols) in enumerate(experiments):
    print(f"\n=======================================================")
    print(f"  [STEP 1: OPTUNA] Individual Tuning for '{exp_name}' ({len(feat_cols)} Features)")
    print(f"=======================================================")

    X_feat_matrix = df[feat_cols].to_numpy(dtype=np.float32)
    TUNING_SEED = SEED + exp_idx * 100

    Xva_t = X_feat_matrix[idx_va]
    Yva_t = y_all[idx_va]

    def objective_exp(trial):
        param = {
            'objective': 'binary:logistic',
            'eval_metric': 'aucpr',
            'booster': 'gbtree',
            'tree_method': 'hist',
            'device': DEVICE,
            'random_state': TUNING_SEED,
            'n_jobs': -1,
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', max(1.0, 0.05 * raw_ratio), min(2000.0, 2.0 * raw_ratio), log=True),
            'n_estimators': trial.suggest_int('n_estimators', 100, 600, step=50),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.85, 1.0),
            'gamma': trial.suggest_float('gamma', 0.1, 5.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        }

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=TUNING_SEED)
        cv_scores = []
        for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(idx_tr, y_all[idx_tr])):
            fold_tr_idx = idx_tr[tr_idx]
            fold_va_idx = idx_tr[val_idx]
            X_tr_fold, Y_tr_fold = X_feat_matrix[fold_tr_idx], y_all[fold_tr_idx]
            X_va_fold, Y_va_fold = X_feat_matrix[fold_va_idx], y_all[fold_va_idx]
            X_tr_sub, Y_tr_sub = get_memory_safe_subsample(X_tr_fold, Y_tr_fold, max_negs=250_000, seed=TUNING_SEED + fold_idx)

            model = xgb.XGBClassifier(**param, early_stopping_rounds=30)
            model.fit(X_tr_sub, Y_tr_sub, eval_set=[(X_va_fold, Y_va_fold)], verbose=False)
            val_probs = model.predict_proba(X_va_fold)[:, 1]
            pr_auc = average_precision_score(Y_va_fold, val_probs)
            cv_scores.append(pr_auc)
            del model, X_tr_fold, Y_tr_fold, X_va_fold, Y_va_fold, X_tr_sub, Y_tr_sub
            gc.collect()
        return float(np.mean(cv_scores))

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=TUNING_SEED))
    study.optimize(objective_exp, n_trials=N_TRIALS)

    exp_params = study.best_trial.params.copy()
    exp_params.update({
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'booster': 'gbtree',
        'tree_method': 'hist',
        'device': DEVICE,
        'random_state': TUNING_SEED,
        'n_jobs': -1,
    })
    print(f"[{exp_name} OPTUNA DONE] Best params: {json.dumps({k: v for k, v in exp_params.items() if k not in ['objective','eval_metric','booster','tree_method','device','n_jobs']}, indent=2)}")

    print(f"\n=======================================================")
    print(f"  [STEP 2: EVALUATION] 5-Fold CV Evaluation for '{exp_name}'")
    print(f"=======================================================")

    val_probs_ensemble = np.zeros(len(Xva_t), dtype=np.float32)
    test_probs_ensemble = np.zeros(len(idx_te), dtype=np.float32)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(idx_tr, y_all[idx_tr])):
        sub_tr_idx = idx_tr[tr_idx]
        sub_va_idx = idx_tr[val_idx]
        X_tr_full_f, Y_tr_full_f = X_feat_matrix[sub_tr_idx], y_all[sub_tr_idx]
        X_va_f, Y_va_f = X_feat_matrix[sub_va_idx], y_all[sub_va_idx]
        X_tr_f, Y_tr_f = get_memory_safe_subsample(X_tr_full_f, Y_tr_full_f, max_negs=500_000, seed=SEED + fold_idx)

        fold_model = xgb.XGBClassifier(**exp_params, early_stopping_rounds=40)
        fold_model.fit(X_tr_f, Y_tr_f, eval_set=[(X_va_f, Y_va_f)], verbose=False)
        val_probs_ensemble += fold_model.predict_proba(Xva_t)[:, 1] / 5.0

        test_batch_size = 1_000_000
        for b_start in range(0, len(idx_te), test_batch_size):
            b_end = min(b_start + test_batch_size, len(idx_te))
            b_idx = idx_te[b_start:b_end]
            b_Xte = X_feat_matrix[b_idx]
            test_probs_ensemble[b_start:b_end] += fold_model.predict_proba(b_Xte)[:, 1] / 5.0

        del fold_model, X_tr_full_f, Y_tr_full_f, X_tr_f, Y_tr_f, X_va_f, Y_va_f
        gc.collect()

    del X_feat_matrix
    gc.collect()

    Yte = y_all[idx_te]

    # Individual Optimal Threshold Search per feature group on holdout validation set
    eval_thr, _ = find_best_f1_threshold(Yva_t, val_probs_ensemble)
    test_preds = (test_probs_ensemble >= eval_thr).astype(int)

    acc  = accuracy_score(Yte, test_preds)
    prec = precision_score(Yte, test_preds, zero_division=0)
    rec  = recall_score(Yte, test_preds, zero_division=0)
    f1   = f1_score(Yte, test_preds, zero_division=0)
    f2   = fbeta_score(Yte, test_preds, beta=2, zero_division=0)
    auc  = roc_auc_score(Yte, test_probs_ensemble)
    pr_auc = average_precision_score(Yte, test_probs_ensemble)
    cm   = confusion_matrix(Yte, test_preds)
    tn, fp, fn, tp = cm.ravel()

    top1000_idx = np.argsort(test_probs_ensemble)[::-1][:1000]
    top1000_hits = int(Yte[top1000_idx].sum())

    top3000_idx = np.argsort(test_probs_ensemble)[::-1][:3000]
    top3000_hits = int(Yte[top3000_idx].sum())

    report = {
        "num_features": len(feat_cols),
        "optimal_threshold": float(eval_thr),
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "f2_score": float(f2),
        "roc_auc": float(auc),
        "pr_auc": float(pr_auc),
        "top1000_hits": top1000_hits,
        "top3000_hits": top3000_hits,
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    }

    ablation_results[exp_name] = report
    print(f"[{exp_name:5s} SUMMARY] F1 = {f1:.6f} | PR-AUC = {pr_auc:.6f} | ROC-AUC = {auc:.6f} | Top1k_Hits = {top1000_hits}명 | Top3k_Hits = {top3000_hits}명 | Rec = {rec:.6f} | Thr = {eval_thr:.4f}")




print("\n=======================================================")
print("       XGBoost FEATURE ABLATION SUMMARY REPORT         ")
print("=======================================================")
print(json.dumps(ablation_results, indent=2))

# Save JSON report to D:\LJH\predict
OUT_JSON_DIR = r"D:\LJH\predict"
if not os.path.exists(r"D:\LJH") and os.path.exists(r"/mnt/d/LJH"):
    OUT_JSON_DIR = r"/mnt/d/LJH/predict"
if not os.path.exists(os.path.dirname(OUT_JSON_DIR)):
    OUT_JSON_DIR = os.path.join(NEW_FOLDER, "predict")

os.makedirs(OUT_JSON_DIR, exist_ok=True)
json_save_path = os.path.join(OUT_JSON_DIR, "result_xgb.json")

with open(json_save_path, "w", encoding="utf-8") as f:
    json.dump(ablation_results, f, indent=2, ensure_ascii=False)

print(f"\n[SAVED] JSON Report successfully saved to: {json_save_path}")





[DEVICE] Detected device for XGBoost: cuda
[LOAD] Loading dataset from: /mnt/d/LJH/data/final_data_100k_64.parquet
[MEMORY OPTIMIZATION] Downcasting feature dtypes to float32/int16...
[DATASET CHECK] Total Rows: 9,999,329 | Positives: 12,340 | Target Imbalance Ratio: 0.123408%
[COST-SENSITIVE] Full Train Set Negative/Positive Ratio = 809.32

  [STEP 1: OPTUNA] Individual Tuning for 'base' (41 Features)
[base OPTUNA DONE] Best params: {
  "scale_pos_weight": 58.163050194986155,
  "n_estimators": 300,
  "learning_rate": 0.2599674562373734,
  "max_depth": 7,
  "min_child_weight": 7,
  "subsample": 0.6946546893018188,
  "colsample_bytree": 0.9529751391522375,
  "gamma": 4.189665792297127,
  "reg_alpha": 0.0011834587074410605,
  "reg_lambda": 1.0013300735281483,
  "random_state": 1
}

  [STEP 2: EVALUATION] 5-Fold CV Evaluation for 'base'
[base  SUMMARY] F1 = 0.358094 | PR-AUC = 0.322834 | ROC-AUC = 0.996479 | Top1k_Hits = 484명 | Top3k_Hits = 846명 | Rec = 0.326850 | Thr = 0.9908

  [STEP 1:

In [1]:
import os, sys, random, json, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve
)

import torch
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
except ImportError:
    print("[ERROR] 'pytorch-tabnet' package is not installed. Please install via: pip install pytorch-tabnet")
    sys.exit(1)

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

# =====================================================
# 1. CONFIG & SEED
# =====================================================
SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

BASE_DIR = r"C:\Users\DIVE_GUEST\LJH\ecommerce_journey"
if not os.path.exists(BASE_DIR) and os.path.exists(r"/mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey"):
    BASE_DIR = r"/mnt/c/Users/DIVE_GUEST/LJH/ecommerce_journey"
if not os.path.exists(BASE_DIR):
    BASE_DIR = r"c:\Users\user\Desktop\마케팅도메인지식기반 구매예측_논문\장바구니 이탈 연구 선행 논문"

NEW_FOLDER = os.path.join(BASE_DIR, "새 폴더")

DATA_DIR = r"/mnt/d/LJH/data"
if not os.path.exists(DATA_DIR) and os.path.exists(r"D:\LJH\data"):
    DATA_DIR = r"D:\LJH\data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = BASE_DIR

PATH_PARQUET = os.path.join(DATA_DIR, "final_data_100k_64.parquet")
if not os.path.exists(PATH_PARQUET):
    PATH_PARQUET = os.path.join(BASE_DIR, "final_data_100k_64.parquet")

N_TRIALS = 15
TEST_FRAC = 0.15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[DEVICE] Detected device for TabNet: {DEVICE}")

# =======================================================
# 2. FEATURE GROUPS (v0 ~ v4) FOR ABLATION STUDY
# =======================================================
cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

cols_v1 = ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']
cols_v2 = ['cal_is_weekend', 'cal_week_of_month', 'feat_dow_shift', 'feat_eoq_bump', 'feat_hour_shift', 'feat_payday_bump']
cols_v3 = ['ctx_tc_open_rate_30d', 'feat_fatigue', 'feat_last_any_hours', 'feat_last_email_hours', 'feat_last_mobile_push_hours', 'u_cadence_std_30d', 'u_click_rate_30d', 'u_open_cnt_30d', 'u_open_rate_30d']
cols_v4 = ['feat_like_last_success', 'feat_path_align', 'feat_topic_novelty', 'topic_N7', 'topic_t_since_hours']

# =====================================================
# 3. LOAD DATASET
# =====================================================
if not os.path.exists(PATH_PARQUET):
    print(f"[ERROR] Parquet file does not exist at {PATH_PARQUET}!")
    sys.exit(1)

print(f"[LOAD] Loading dataset from: {PATH_PARQUET}")
df = pd.read_parquet(PATH_PARQUET)
TARGET = "is_purchased"

# Memory Downcasting
print("[MEMORY OPTIMIZATION] Downcasting feature dtypes to float32/int16...")
for c in df.columns:
    if "float" in str(df[c].dtype):
        df[c] = df[c].astype(np.float32)
    elif "int" in str(df[c].dtype) and c != TARGET:
        df[c] = df[c].astype(np.int16)
gc.collect()

print(f"[DATASET CHECK] Total Rows: {len(df):,d} | Positives: {(df[TARGET]==1).sum():,d} | Target Imbalance Ratio: {df[TARGET].mean()*100:.6f}%")

def get_existing_cols(df, col_list):
    return [c for c in col_list if c in df.columns]

feat_cols_base  = get_existing_cols(df, cols_v0)
feat_cols_v0_v1 = get_existing_cols(df, cols_v0 + cols_v1)
feat_cols_v0_v2 = get_existing_cols(df, cols_v0 + cols_v2)
feat_cols_v0_v3 = get_existing_cols(df, cols_v0 + cols_v3)
feat_cols_v0_v4 = get_existing_cols(df, cols_v0 + cols_v4)
feat_cols_all   = get_existing_cols(df, cols_v0 + cols_v1 + cols_v2 + cols_v3 + cols_v4)

# =====================================================
# 4. HELPER FUNCTIONS
# =====================================================
def find_best_f2_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    if len(thresholds) == 0:
        return 0.5, 0.0
    f2_scores = (5 * precisions * recalls) / np.maximum(4 * precisions + recalls, 1e-10)
    best_idx = np.argmax(f2_scores)
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f2 = f2_scores[best_idx]
    return float(best_thr), float(best_f2)

def find_best_f1_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    if len(thresholds) == 0:
        return 0.5, 0.0
    f1_scores = 2 * (precisions * recalls) / np.maximum(precisions + recalls, 1e-10)
    best_idx = np.argmax(f1_scores)
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f1 = f1_scores[best_idx]
    return float(best_thr), float(best_f1)

def get_memory_safe_subsample(X_in, Y_in, max_negs=250_000, seed=SEED):
    pos_idx = np.where(Y_in == 1)[0]
    neg_idx = np.where(Y_in == 0)[0]
    if len(neg_idx) <= max_negs:
        return X_in, Y_in
    rng_sub = np.random.RandomState(seed)
    sub_neg_idx = rng_sub.choice(neg_idx, size=max_negs, replace=False)
    comb_idx = np.concatenate([pos_idx, sub_neg_idx])
    rng_sub.shuffle(comb_idx)
    return X_in[comb_idx], Y_in[comb_idx]

# =====================================================
# 5. INDIVIDUAL OPTUNA & ABLATION EVALUATION FOR TABNET
# =====================================================
ablation_results = {}

experiments = [
    ("base",  feat_cols_base),
    ("v0_v1", feat_cols_v0_v1),
    ("v0_v2", feat_cols_v0_v2),
    ("v0_v3", feat_cols_v0_v3),
    ("v0_v4", feat_cols_v0_v4),
    ("all",   feat_cols_all),
]

all_indices = np.arange(len(df))
y_all = df[TARGET].values.astype(np.int32)

idx_tr_full, idx_te = train_test_split(all_indices, test_size=TEST_FRAC, stratify=y_all, random_state=SEED)
idx_tr, idx_va = train_test_split(idx_tr_full, test_size=0.17647, stratify=y_all[idx_tr_full], random_state=SEED)

raw_ratio = (y_all[idx_tr] == 0).sum() / float(max(1, (y_all[idx_tr] == 1).sum()))
print(f"[COST-SENSITIVE] Full Train Set Negative/Positive Ratio = {raw_ratio:.2f}")

for exp_idx, (exp_name, feat_cols) in enumerate(experiments):
    print(f"\n=======================================================")
    print(f"  [STEP 1: OPTUNA] Individual Tuning for '{exp_name}' ({len(feat_cols)} Features)")
    print(f"=======================================================")

    raw_feat_matrix = df[feat_cols].fillna(0).to_numpy(dtype=np.float32)
    scaler = StandardScaler()
    X_feat_matrix = scaler.fit_transform(raw_feat_matrix)

    TUNING_SEED = SEED + exp_idx * 100

    Xva_t = X_feat_matrix[idx_va]
    Yva_t = y_all[idx_va]

    def objective_exp(trial):
        n_d = trial.suggest_int('n_d', 8, 64, step=8)
        n_a = n_d
        n_steps = trial.suggest_int('n_steps', 3, 6)
        gamma = trial.suggest_float('gamma', 1.0, 2.0)
        lambda_sparse = trial.suggest_float('lambda_sparse', 1e-4, 1e-2, log=True)
        lr = trial.suggest_float('lr', 1e-3, 5e-2, log=True)
        mask_type = trial.suggest_categorical('mask_type', ['sparsemax', 'entmax'])
        batch_size = trial.suggest_categorical('batch_size', [1024, 2048, 4096])
        virtual_batch_size = trial.suggest_categorical('virtual_batch_size', [128, 256])

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=TUNING_SEED)
        cv_scores = []

        for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(idx_tr, y_all[idx_tr])):
            fold_tr_idx = idx_tr[tr_idx]
            fold_va_idx = idx_tr[val_idx]
            X_tr_fold, Y_tr_fold = X_feat_matrix[fold_tr_idx], y_all[fold_tr_idx]
            X_va_fold, Y_va_fold = X_feat_matrix[fold_va_idx], y_all[fold_va_idx]
            X_tr_sub, Y_tr_sub = get_memory_safe_subsample(X_tr_fold, Y_tr_fold, max_negs=250_000, seed=TUNING_SEED + fold_idx)

            model = TabNetClassifier(
                n_d=n_d,
                n_a=n_a,
                n_steps=n_steps,
                gamma=gamma,
                lambda_sparse=lambda_sparse,
                optimizer_fn=torch.optim.Adam,
                optimizer_params=dict(lr=lr),
                mask_type=mask_type,
                scheduler_fn=torch.optim.lr_scheduler.StepLR,
                scheduler_params=dict(step_size=10, gamma=0.9),
                device_name=DEVICE,
                verbose=0
            )

            model.fit(
                X_train=X_tr_sub, y_train=Y_tr_sub,
                eval_set=[(X_va_fold, Y_va_fold)],
                eval_name=['val'],
                eval_metric=['auc'],
                max_epochs=25,
                patience=8,
                batch_size=batch_size,
                virtual_batch_size=virtual_batch_size,
                weights=1,
                drop_last=False
            )

            val_probs = model.predict_proba(X_va_fold)[:, 1]
            pr_auc = average_precision_score(Y_va_fold, val_probs)
            cv_scores.append(pr_auc)

            del model, X_tr_fold, Y_tr_fold, X_va_fold, Y_va_fold, X_tr_sub, Y_tr_sub
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

        return float(np.mean(cv_scores))

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=TUNING_SEED))
    study.optimize(objective_exp, n_trials=N_TRIALS)

    best_params = study.best_trial.params.copy()
    print(f"[{exp_name} OPTUNA DONE] Best params: {json.dumps(best_params, indent=2)}")

    print(f"\n=======================================================")
    print(f"  [STEP 2: EVALUATION] 5-Fold CV Evaluation for '{exp_name}'")
    print(f"=======================================================")

    val_probs_ensemble = np.zeros(len(Xva_t), dtype=np.float32)
    test_probs_ensemble = np.zeros(len(idx_te), dtype=np.float32)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(idx_tr, y_all[idx_tr])):
        sub_tr_idx = idx_tr[tr_idx]
        sub_va_idx = idx_tr[val_idx]
        X_tr_full_f, Y_tr_full_f = X_feat_matrix[sub_tr_idx], y_all[sub_tr_idx]
        X_va_f, Y_va_f = X_feat_matrix[sub_va_idx], y_all[sub_va_idx]
        X_tr_f, Y_tr_f = get_memory_safe_subsample(X_tr_full_f, Y_tr_full_f, max_negs=500_000, seed=SEED + fold_idx)

        n_d = best_params['n_d']
        fold_model = TabNetClassifier(
            n_d=n_d,
            n_a=n_d,
            n_steps=best_params['n_steps'],
            gamma=best_params['gamma'],
            lambda_sparse=best_params['lambda_sparse'],
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=best_params['lr']),
            mask_type=best_params['mask_type'],
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            scheduler_params=dict(step_size=10, gamma=0.9),
            device_name=DEVICE,
            verbose=0
        )

        fold_model.fit(
            X_train=X_tr_f, y_train=Y_tr_f,
            eval_set=[(X_va_f, Y_va_f)],
            eval_name=['val'],
            eval_metric=['auc'],
            max_epochs=35,
            patience=10,
            batch_size=best_params['batch_size'],
            virtual_batch_size=best_params['virtual_batch_size'],
            weights=1,
            drop_last=False
        )

        val_probs_ensemble += fold_model.predict_proba(Xva_t)[:, 1] / 5.0

        test_batch_size = 500_000
        for b_start in range(0, len(idx_te), test_batch_size):
            b_end = min(b_start + test_batch_size, len(idx_te))
            b_idx = idx_te[b_start:b_end]
            b_Xte = X_feat_matrix[b_idx]
            test_probs_ensemble[b_start:b_end] += fold_model.predict_proba(b_Xte)[:, 1] / 5.0

        del fold_model, X_tr_full_f, Y_tr_full_f, X_tr_f, Y_tr_f, X_va_f, Y_va_f
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    del X_feat_matrix, raw_feat_matrix
    gc.collect()

    Yte = y_all[idx_te]

    # Threshold Search on holdout validation set
    eval_thr, _ = find_best_f1_threshold(Yva_t, val_probs_ensemble)
    test_preds = (test_probs_ensemble >= eval_thr).astype(int)

    acc  = accuracy_score(Yte, test_preds)
    prec = precision_score(Yte, test_preds, zero_division=0)
    rec  = recall_score(Yte, test_preds, zero_division=0)
    f1   = f1_score(Yte, test_preds, zero_division=0)
    f2   = fbeta_score(Yte, test_preds, beta=2, zero_division=0)
    auc  = roc_auc_score(Yte, test_probs_ensemble)
    pr_auc = average_precision_score(Yte, test_probs_ensemble)
    cm   = confusion_matrix(Yte, test_preds)
    tn, fp, fn, tp = cm.ravel()

    top1000_idx = np.argsort(test_probs_ensemble)[::-1][:1000]
    top1000_hits = int(Yte[top1000_idx].sum())

    top3000_idx = np.argsort(test_probs_ensemble)[::-1][:3000]
    top3000_hits = int(Yte[top3000_idx].sum())

    report = {
        "num_features": len(feat_cols),
        "optimal_threshold": float(eval_thr),
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "f2_score": float(f2),
        "roc_auc": float(auc),
        "pr_auc": float(pr_auc),
        "top1000_hits": top1000_hits,
        "top3000_hits": top3000_hits,
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    }

    ablation_results[exp_name] = report
    print(f"[{exp_name:5s} SUMMARY] F1 = {f1:.6f} | PR-AUC = {pr_auc:.6f} | ROC-AUC = {auc:.6f} | Top1k_Hits = {top1000_hits}명 | Top3k_Hits = {top3000_hits}명 | Rec = {rec:.6f} | Thr = {eval_thr:.4f}")

print("\n=======================================================")
print("        TabNet FEATURE ABLATION SUMMARY REPORT         ")
print("=======================================================")
print(json.dumps(ablation_results, indent=2))

# Save JSON report
OUT_JSON_DIR = r"D:\LJH\predict"
if not os.path.exists(r"D:\LJH") and os.path.exists(r"/mnt/d/LJH"):
    OUT_JSON_DIR = r"/mnt/d/LJH/predict"
if not os.path.exists(os.path.dirname(OUT_JSON_DIR)):
    OUT_JSON_DIR = os.path.join(NEW_FOLDER, "predict")

os.makedirs(OUT_JSON_DIR, exist_ok=True)
json_save_path = os.path.join(OUT_JSON_DIR, "result_tabnet.json")

with open(json_save_path, "w", encoding="utf-8") as f:
    json.dump(ablation_results, f, indent=2, ensure_ascii=False)

print(f"\n[SAVED] JSON Report successfully saved to: {json_save_path}")

[DEVICE] Detected device for TabNet: cuda
[LOAD] Loading dataset from: /mnt/d/LJH/data/final_data_100k_64.parquet
[MEMORY OPTIMIZATION] Downcasting feature dtypes to float32/int16...
[DATASET CHECK] Total Rows: 9,999,329 | Positives: 12,340 | Target Imbalance Ratio: 0.123408%
[COST-SENSITIVE] Full Train Set Negative/Positive Ratio = 809.32

  [STEP 1: OPTUNA] Individual Tuning for 'base' (41 Features)
Stop training because you reached max_epochs = 25 with best_epoch = 19 and best_val_auc = 0.99375
Stop training because you reached max_epochs = 25 with best_epoch = 19 and best_val_auc = 0.99326
Stop training because you reached max_epochs = 25 with best_epoch = 18 and best_val_auc = 0.99173
Stop training because you reached max_epochs = 25 with best_epoch = 19 and best_val_auc = 0.99359

Early stopping occurred at epoch 22 with best_epoch = 14 and best_val_auc = 0.99323
Stop training because you reached max_epochs = 25 with best_epoch = 18 and best_val_auc = 0.99271
Stop training becaus